In [0]:
%pip install kagglehub
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS proyecto_final_mlops.telco_database;

CREATE VOLUME IF NOT EXISTS proyecto_final_mlops.telco_database.raw_data;

In [0]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "muhammadshahidazeem/customer-churn-dataset"
)

print(dataset_path)

100%|██████████| 6.66M/6.66M [00:00<00:00, 54.8MB/s]

Extracting files...


/home/spark-a38edea7-a61e-47af-b679-37/.cache/kagglehub/datasets/muhammadshahidazeem/customer-churn-dataset/versions/1


In [0]:
import os

files = os.listdir(dataset_path)

training_file = [f for f in files if "train" in f.lower()][0]
test_file = [f for f in files if "test" in f.lower()][0]

print("Training:", training_file)
print("Testing:", test_file)

Training: customer_churn_dataset-training-master.csv
Testing: customer_churn_dataset-testing-master.csv


In [0]:
import shutil

volume_path = "/Volumes/proyecto_final_mlops/telco_database/raw_data"

training_path = f"{volume_path}/training.csv"
test_path = f"{volume_path}/testing.csv"

shutil.copy2(
    os.path.join(dataset_path, training_file),
    training_path
)

shutil.copy2(
    os.path.join(dataset_path, test_file),
    test_path
)

print("Archivos cargados al Volume")

Archivos cargados al Volume


In [0]:
df_training = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(training_path)
)

df_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(test_path)
)

print("Training:", df_training.count())
print("Testing:", df_test.count())

display(df_training.limit(5))

Training: 440833
Testing: 64374


CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
2,30,Female,39,14,5,18,Standard,Annual,932.0,17,1
3,65,Female,49,1,10,8,Basic,Monthly,557.0,6,1
4,55,Female,14,4,6,18,Basic,Quarterly,185.0,3,1
5,58,Male,38,21,7,7,Standard,Monthly,396.0,29,1
6,23,Male,32,20,5,8,Basic,Monthly,617.0,20,1


In [0]:
from pyspark.sql.functions import col

# Rename columns to replace spaces with underscores
df_training_clean = df_training.select(
    [col(c).alias(c.replace(" ", "_")) for c in df_training.columns]
)

df_test_clean = df_test.select(
    [col(c).alias(c.replace(" ", "_")) for c in df_test.columns]
)

df_training_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "proyecto_final_mlops.telco_database.churn_dataset_training_data"
    )

df_test_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "proyecto_final_mlops.telco_database.churn_dataset_test_data"
    )